# REHABELT (PREDICTION MODEL)

###### FORCE INSTALL PYTHON 3.9

In [ ]:
%%bash
# 1. Download Miniconda (Python 3.9 Installer)
MINICONDA_INSTALLER_SCRIPT=Miniconda3-py39_4.12.0-Linux-x86_64.sh
MINICONDA_PREFIX=/usr/local
wget https://repo.anaconda.com/miniconda/$MINICONDA_INSTALLER_SCRIPT

# 2. Install it over the current Python
chmod +x $MINICONDA_INSTALLER_SCRIPT
./$MINICONDA_INSTALLER_SCRIPT -b -f -p $MINICONDA_PREFIX

# 3. Delete the installer to save space
rm $MINICONDA_INSTALLER_SCRIPT

PREFIX=/usr/local
Unpacking payload ...
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - _libgcc_mutex==0.1=main
    - _openmp_mutex==4.5=1_gnu
    - brotlipy==0.7.0=py39h27cfd23_1003
    - ca-certificates==2022.3.29=h06a4308_1
    - certifi==2021.10.8=py39h06a4308_2
    - cffi==1.15.0=py39hd667e15_1
    - charset-normalizer==2.0.4=pyhd3eb1b0_0
    - colorama==0.4.4=pyhd3eb1b0_0
    - conda-content-trust==0.1.1=pyhd3eb1b0_0
    - conda-package-handling==1.8.1=py39h7f8727e_0
    - conda==4.12.0=py39h06a4308_0
    - cryptography==36.0.0=py39h9ce1e76_0
    - idna==3.3=pyhd3eb1b0_0
    - ld_impl_linux-64==2.35.1=h7274673_9
    - libffi==3.3=he6710b0_2
    - libgcc-ng==9.3.0=h5101ec6_17
    - libgomp==9.3.0=h5101ec6_17
    - libstdcxx-ng==9.3.0=hd4cf53a_17
    - ncurses==6.3=h7f8727e_2
    - openssl==1.1.1n=h7f8727e_0
    - pip==21.2.4=py39h06a4308_0
    - pycosat==0.6.3=py39h27cfd23_0
    - pycparser==2.21=pyhd3

--2026-01-24 11:17:11--  https://repo.anaconda.com/miniconda/Miniconda3-py39_4.12.0-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 76607678 (73M) [application/x-sh]
Saving to: ‘Miniconda3-py39_4.12.0-Linux-x86_64.sh’

     0K .......... .......... .......... .......... ..........  0% 48.4M 2s
    50K .......... .......... .......... .......... ..........  0% 3.71M 11s
   100K .......... .......... .......... .......... ..........  0%  203M 7s
   150K .......... .......... .......... .......... ..........  0%  210M 5s
   200K .......... .......... .......... .......... ..........  0% 6.90M 6s
   250K .......... .......... .......... .......... ..........  0%  197M 5s
   300K .......... .......... .......... .......... ..........  0%  257M 5s
   350K .......... .......... .

##### DOWNGRADE TENSORFLOW (Fixes "Version 9" Crash)

In [ ]:
!pip uninstall -y tensorflow
!pip install tensorflow==2.10.0

Found existing installation: tensorflow 2.10.0
Uninstalling tensorflow-2.10.0:
  Successfully uninstalled tensorflow-2.10.0
  Using cached tensorflow-2.10.0-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (578.1 MB)


###### MOUNT GOOGLE DRIVE

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# Verify the main folder exists
folder_path = '/content/drive/MyDrive/rehabelt_data'
if os.path.exists(folder_path):
    print(f"\nSUCCESS: Found main folder: {folder_path}")
else:
    print(f"\nERROR: Main folder '{folder_path}' not found.")

Mounted at /content/drive

SUCCESS: Found main folder: /content/drive/MyDrive/rehabelt_data


##### MAIN SCRIPT

In [11]:
%%bash

# 1. SETUP ENVIRONMENT
rm -rf legacy_env Miniconda3-py39_4.12.0-Linux-x86_64.sh
wget -q https://repo.anaconda.com/miniconda/Miniconda3-py39_4.12.0-Linux-x86_64.sh
chmod +x Miniconda3-py39_4.12.0-Linux-x86_64.sh
./Miniconda3-py39_4.12.0-Linux-x86_64.sh -b -f -p ./legacy_env > /dev/null
./legacy_env/bin/python -m pip install -q tensorflow==2.10.0 pandas "numpy<2" scikit-learn matplotlib seaborn scipy

# 2. WRITE BUILDER SCRIPT (DENSE VERSION)
cat > builder.py <<EOF
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from sklearn.utils import class_weight
from sklearn.preprocessing import StandardScaler
from scipy.signal import resample
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# CONFIGURATION
INPUT_FOLDER = '/content/drive/MyDrive/rehabelt_data'
OUTPUT_FOLDER = '/content/drive/MyDrive/Commission_Files'
KFALL_HZ = 100
MY_DEVICE_HZ = 50
TARGET_WINDOW_SIZE = 100

if not os.path.exists(OUTPUT_FOLDER): os.makedirs(OUTPUT_FOLDER)

# LOAD DATA
csv_files = glob.glob(os.path.join(INPUT_FOLDER, "**/*.csv"), recursive=True)
X_all, y_all = [], []

print(f"Processing {len(csv_files)} files...")

for file in csv_files:
    try:
        df = pd.read_csv(file)
        data = df.iloc[:, 0:6].values
        name = os.path.basename(file).upper()

        label = -1
        if "T18" in name or "T19" in name or "SUPINE" in name: label = 2
        elif "T13" in name or "T14" in name or "SIT" in name: label = 1
        elif "T06" in name or "T07" in name or "WALK" in name: label = 0

        if label == -1: continue

        if np.mean(np.abs(data[:, 0:3])) < 5.0:
            data[:, 0:3] = data[:, 0:3] * 9.8
            current_hz = KFALL_HZ
        else:
            current_hz = MY_DEVICE_HZ

        required_samples = int(current_hz * (TARGET_WINDOW_SIZE / KFALL_HZ))
        step = int(required_samples * 0.5)

        if len(data) >= required_samples:
            for i in range(0, len(data) - required_samples, step):
                raw_window = data[i : i + required_samples]
                if len(raw_window) != TARGET_WINDOW_SIZE:
                    window = resample(raw_window, TARGET_WINDOW_SIZE)
                else:
                    window = raw_window

                if label == 0 or np.sum(np.std(window, axis=0)) > 2.0:
                    X_all.append(window)
                    y_all.append(label)
    except: pass

X = np.array(X_all)
y = np.array(y_all)
print(f"Loaded {len(X)} windows.")

# NORMALIZE
scaler = StandardScaler()
X_flat = X.reshape(-1, 6)
X_scaled = scaler.fit_transform(X_flat).reshape(X.shape)

n_noise = int(len(X) / 3)
X_noise = []
for i in range(n_noise):
    X_noise.append(np.random.normal(0, 1, (100, 6)))

X_final = np.concatenate((X_scaled, np.array(X_noise)), axis=0)
y_final = np.concatenate((y, np.full(n_noise, 3)), axis=0)

# FLATTEN DATA FOR DENSE NETWORK
X_final = X_final.reshape(X_final.shape[0], -1)
print(f"Data Flattened to: {X_final.shape}")

y_integers = y_final.astype(int)
class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_integers), y=y_integers)
cw_dict = dict(enumerate(class_weights))

y_encoded = to_categorical(y_final, 4)
X_train, X_val, y_train, y_val = train_test_split(X_final, y_encoded, test_size=0.2, stratify=y_encoded)

# DENSE MODEL
model = Sequential([
    Dense(128, activation='relu', input_shape=(600,)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("Starting Training...")
model.fit(X_train, y_train, epochs=90, batch_size=32, validation_data=(X_val, y_val), verbose=1, class_weight=cw_dict)

# EVALUATION METRICS & MATRIX
y_pred_probs = model.predict(X_val)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_val, axis=1)
class_names = ['Walking', 'Sit-Stand', 'Supine-Sit', 'Unrecognized']

print("DETAILED PERFORMANCE REPORT")

# Report (F1, Precision, Recall)
print(classification_report(y_true, y_pred, target_names=class_names))

print("Mean Average Precision (mAP) per Category:")
for i, name in enumerate(class_names):
    try:
        ap = average_precision_score(y_val[:, i], y_pred_probs[:, i])
        print(f"   {name:15s}: {ap:.4f}")
    except: pass

mAP = average_precision_score(y_val, y_pred_probs, average="macro")
print(f"Overall mAP (Macro): {mAP:.4f}")

# Save Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
save_path = os.path.join(OUTPUT_FOLDER, 'confusion_matrix.png')
plt.savefig(save_path, bbox_inches='tight')
print(f"Confusion Matrix saved to: {save_path}")

# SAVE HEADER
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_model = converter.convert()

# OP CHECKER
try:
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()
    ops = set()
    for i in range(interpreter.get_output_details()[0]['index']):
        try:
            op = interpreter._get_node_and_registration(i)[1]
            ops.add(op.builtin_code)
        except: pass
    print("MODEL OPERATIONS DETECTED:", ops)
except: pass

def hex_to_c_array(data):
    c_str = '#include <cstdint>\\n'
    c_str += f'const unsigned int model_data_len = {len(data)};\\n'
    c_str += f'alignas(16) const unsigned char model_data[] = {{\\n  '
    hex_array = [f'0x{val:02x}' for val in data]
    for i in range(0, len(hex_array), 12):
        c_str += ', '.join(hex_array[i:i+12])
        if i + 12 < len(hex_array): c_str += ",\\n  "
    c_str += '\\n};\\n'
    return c_str

with open(os.path.join(OUTPUT_FOLDER, 'model_data.h'), "w") as f:
    f.write(hex_to_c_array(tflite_model))

print(f"Calibration Values:")
print(f"MEAN: {scaler.mean_}")
print(f"SCALE: {scaler.scale_}")
EOF

# RUN
MPLBACKEND=Agg ./legacy_env/bin/python builder.py

Processing 699 files...
Loaded 21413 windows.
Data Flattened to: (28550, 600)
Starting Training...
Epoch 1/90
714/714 [==============================] - 4s 4ms/step - loss: 0.6458 - accuracy: 0.7422 - val_loss: 0.2505 - val_accuracy: 0.8954
Epoch 2/90
714/714 [==============================] - 4s 5ms/step - loss: 0.3685 - accuracy: 0.8623 - val_loss: 0.1954 - val_accuracy: 0.9268
Epoch 3/90
714/714 [==============================] - 3s 4ms/step - loss: 0.3025 - accuracy: 0.8875 - val_loss: 0.1879 - val_accuracy: 0.9256
Epoch 4/90
714/714 [==============================] - 3s 4ms/step - loss: 0.2722 - accuracy: 0.8984 - val_loss: 0.1713 - val_accuracy: 0.9263
Epoch 5/90
714/714 [==============================] - 3s 4ms/step - loss: 0.2596 - accuracy: 0.9025 - val_loss: 0.1661 - val_accuracy: 0.9312
Epoch 6/90
714/714 [==============================] - 3s 4ms/step - loss: 0.2461 - accuracy: 0.9065 - val_loss: 0.1706 - val_accuracy: 0.9266
Epoch 7/90
714/714 [=============================

2026-01-30 07:23:46.593993: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-30 07:23:46.796983: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-01-30 07:23:46.797025: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-01-30 07:23:46.836856: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-30 07:23:47.886814: W tensorflow/stream_executor/platform/de